# MLE debug — re-evaluate Chisq + MLE fits under the MLE objective

Flow: load behavior + both fit-result pickles, then for a chosen subject
feed BOTH its Chisq-fitted and MLE-fitted parameter sets through
`evaluate_neg_loglik` to get two per-trial MLE diagnostic dataframes:

- `mle_chisq_df` — chisq-fitted params evaluated under MLE
- `mle_mle_df`   — mle-fitted params evaluated under MLE

Then compare them trial-by-trial to find which trials disagree the
most. Use it when the Chisq fit looks better visually but the MLE
objective disagrees.

## Enable loading from relative packages

In [ ]:
%load_ext autoreload
%autoreload 2
if "PKG" not in globals():
    root_parent_level = 3
    import importlib, sys, pathlib
    PKG = %pwd
    PKG = pathlib.Path(PKG)
    root = PKG
    full_pkg = f"{root.name}"
    for _ in range(root_parent_level):
        root = root.parent
        full_pkg = f"{root.name}.{full_pkg}"
        MODULE_PATH = str(root / "__init__.py")
        MODULE_NAME = f"{root.name}"
        spec = importlib.util.spec_from_file_location(MODULE_NAME, MODULE_PATH)
        module = importlib.util.module_from_spec(spec)
        sys.modules[spec.name] = module
        spec.loader.exec_module(module)
    __package__ = full_pkg

## Load all mice data

Identical to `rlmodel_interactive.ipynb`: this creates the `df_behavior`
global that every other cell consumes.

In [ ]:
# Behavior-dataframe prep lives in model/mle_reeval.py (shared with
# rlmodel_compare.ipynb): loadDF → calcAvgRewardRate → _reduceDFSize →
# _extendTrials.
from ..model.mle_reeval import prepare_behavior_df

df_behavior = prepare_behavior_df("../../../data/behavior/df_behavior.pkl")

## Load fit results (MLE + Chisq)

Creates the `fit_results_by_mode` global with the same shape as in
`rlmodel_interactive.ipynb`.

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

RESULT_DIR = Path("../../../data/RLModel")
FIT_MODES = ("mle", "chisq")


def _load_pickle(path):
    with path.open("rb") as f:
        return pickle.load(f)


def loadFitResults(result_dir=RESULT_DIR, all_subjects=None):
    fit_results_by_mode = {mode: {} for mode in FIT_MODES}
    for mode in FIT_MODES:
        for fit_path in sorted(result_dir.glob(f"{mode}_*.pkl")):
            print("Processing:", fit_path)
            loaded = _load_pickle(fit_path)
            fit_results_by_mode[mode][fit_path.name] = loaded
            if all_subjects is not None:
                missing = all_subjects - set(loaded.keys())
                if missing:
                    print("  Missing subjects:", missing)
    return fit_results_by_mode


fit_results_by_mode = loadFitResults(
    all_subjects=set(df_behavior.Name.unique()))

## Helpers — parse filename + re-evaluate any fit under MLE

`fitted_params_to_mle_df` works for both `chisq_*.pkl` and `mle_*.pkl`
variants — the filename's prefix selects which `fit_results_by_mode`
bucket to read from. The optional `lapse_rate_override` lets the driver
force a specific λ value on either fit type (most useful for forcing the
chisq-fitted params to use the same λ as the mle-fitted params for a
fair comparison).

In [ ]:
# Filename parsing + MLE re-evaluation live in model/mle_reeval.py
# (shared with rlmodel_compare.ipynb). `fitted_params_to_mle_df` returns
# (mle_df, eval_result, model_config, params).
from ..model.mle_reeval import parse_fit_filename, fitted_params_to_mle_df
from ..model.mle_likelihood import LOGLIK_FLOOR
import numpy as np
import pandas as pd


def list_variants(fit_results_by_mode, fit_mode, subject_name=None):
    """Variant filenames for the given fit_mode, optionally filtered to
    those that contain `subject_name`."""
    out = []
    for filename, fit_dict in fit_results_by_mode.get(fit_mode, {}).items():
        if subject_name is None or subject_name in fit_dict:
            out.append(filename)
    return out


def list_subjects(fit_results_by_mode, variant_filename):
    """Subjects that have a fit in `variant_filename`."""
    fit_mode = parse_fit_filename(variant_filename).fit_mode
    return sorted(fit_results_by_mode[fit_mode][variant_filename].keys())


def _evaluate_one(subject_name, variant_filename, label,
                  df_behavior, fit_results_by_mode, mle_terminal_c,
                  lapse_rate_override):
    mle_df, eval_res, config, params = fitted_params_to_mle_df(
        subject_name, variant_filename, df_behavior, fit_results_by_mode,
        mle_terminal_c=mle_terminal_c,
        lapse_rate_override=lapse_rate_override)
    effective_lapse = float(params.get("LAPSE_RATE", 0.0))
    lapse_source = ("overridden" if lapse_rate_override is not None
                    else "from fit")
    print(f"[{label}] variant         : {variant_filename}")
    print(f"[{label}] LAPSE_RATE used : {effective_lapse:.6f} "
          f"({lapse_source})")
    print(f"[{label}] neg_loglik (MLE): {eval_res.neg_loglik:.3f}")
    print(f"[{label}] n_trials_loss   : {eval_res.n_trials_loss} / "
          f"{eval_res.n_trials_total}")
    print()
    return mle_df, eval_res, config, params


# ----- Single-df inspection ---------------------------------------------

def summarize_trial_losses(mle_df, top_n=20):
    """One-shot per-trial diagnostic: floor count, by-class loglik, worst trials."""
    valid = mle_df[mle_df["mle_valid_for_loss"].astype(bool)].copy()
    no_choice = valid[valid["ChoiceLeft"].isna()]
    choice = valid[valid["ChoiceLeft"].notna()]
    floor_log = float(np.log(LOGLIK_FLOOR))
    n_floor = int((valid["mle_loglik"] <= floor_log * 0.99).sum())

    print(f"valid trials       : {len(valid):>6d}")
    print(f"  choice trials    : {len(choice):>6d}  "
          f"mean_loglik = {choice['mle_loglik'].mean():>+9.3f}")
    if len(no_choice):
        print(f"  no-choice trials : {len(no_choice):>6d}  "
              f"mean_loglik = {no_choice['mle_loglik'].mean():>+9.3f}")
    else:
        print(f"  no-choice trials : 0")
    pct = 100.0 * n_floor / max(len(valid), 1)
    print(f"trials at floor    : {n_floor:>6d}  ({pct:.1f}%)")
    print(f"total neg_loglik   : {-valid['mle_loglik'].sum():>10.3f}")

    cols = [c for c in [
        "TrialNumber", "DV", "ChoiceLeft", "ChoiceCorrect",
        "calcStimulusTime", "mle_z", "mle_mu", "mle_sigma",
        "mle_decision_time_observed", "mle_choice_prob_or_density",
        "mle_survival_at_tmax", "mle_loglik",
    ] if c in mle_df.columns]
    return valid.nsmallest(top_n, "mle_loglik")[cols]


def plot_trial_loss_histogram(mle_df, bins=60, title_suffix=""):
    """Per-trial loglik histogram split by choice / no-choice."""
    import matplotlib.pyplot as plt
    valid = mle_df[mle_df["mle_valid_for_loss"].astype(bool)]
    choice = valid[valid["ChoiceLeft"].notna()]["mle_loglik"].to_numpy()
    no_choice = valid[valid["ChoiceLeft"].isna()]["mle_loglik"].to_numpy()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(choice, bins=bins, alpha=0.7, label=f"choice (n={len(choice)})")
    if len(no_choice):
        ax.hist(no_choice, bins=bins, alpha=0.7,
                label=f"no-choice (n={len(no_choice)})")
    ax.axvline(float(np.log(LOGLIK_FLOOR)), color="r", ls="--",
               label="log(LOGLIK_FLOOR)")
    ax.set_xlabel("per-trial mle_loglik")
    ax.set_ylabel("count")
    ax.set_title(f"Per-trial loglik distribution{title_suffix}")
    ax.legend()
    fig.tight_layout()
    return fig


def loss_by_rt_bin(mle_df, n_bins=10):
    v = mle_df[mle_df["mle_valid_for_loss"].astype(bool)
               & mle_df["ChoiceLeft"].notna()].copy()
    v["rt_bin"] = pd.qcut(v["calcStimulusTime"], n_bins, duplicates="drop")
    return v.groupby("rt_bin", observed=True)["mle_loglik"].agg(
        ["count", "mean", "min"])


def loss_by_dv(mle_df, n_bins=8):
    v = mle_df[mle_df["mle_valid_for_loss"].astype(bool)
               & mle_df["ChoiceLeft"].notna()].copy()
    v["dv_bin"] = pd.qcut(v["DV"], n_bins, duplicates="drop")
    return v.groupby("dv_bin", observed=True)["mle_loglik"].agg(
        ["count", "mean", "min"])


# ----- Two-df comparison ------------------------------------------------

def compare_loglik_per_trial(mle_chisq_df, mle_mle_df):
    """Side-by-side per-trial loglik DataFrame.

    Both DFs come from `fitted_params_to_mle_df` for the same subject and
    therefore share the same row index — index-based subtraction is exact.
    Returns a frame with `mle_chisq_loglik`, `mle_mle_loglik`,
    `delta_loglik = mle_mle_loglik - mle_chisq_loglik`. Positive `delta`
    means the MLE-fitted parameters fit that trial better.
    """
    common = [c for c in (
        "TrialNumber", "DV", "ChoiceLeft", "ChoiceCorrect",
        "calcStimulusTime") if c in mle_chisq_df.columns]
    out = mle_chisq_df[common].copy()
    out["mle_chisq_loglik"] = mle_chisq_df["mle_loglik"]
    out["mle_mle_loglik"] = mle_mle_df["mle_loglik"]
    out["delta_loglik"] = (
        mle_mle_df["mle_loglik"] - mle_chisq_df["mle_loglik"])
    out["mle_chisq_valid"] = mle_chisq_df["mle_valid_for_loss"]
    out["mle_mle_valid"] = mle_mle_df["mle_valid_for_loss"]
    out["mle_chisq_z"] = mle_chisq_df["mle_z"]
    out["mle_mle_z"] = mle_mle_df["mle_z"]
    out["mle_chisq_mu"] = mle_chisq_df["mle_mu"]
    out["mle_mle_mu"] = mle_mle_df["mle_mu"]
    out["mle_chisq_sigma"] = mle_chisq_df["mle_sigma"]
    out["mle_mle_sigma"] = mle_mle_df["mle_sigma"]
    return out


def summarize_comparison(mle_chisq_df, mle_mle_df, top_n=10):
    """Print head-to-head comparison + worst trials in each direction.
    Returns the full comparison DataFrame for further filtering."""
    cmp = compare_loglik_per_trial(mle_chisq_df, mle_mle_df)
    both_valid = cmp[cmp["mle_chisq_valid"] & cmp["mle_mle_valid"]].copy()
    delta = both_valid["delta_loglik"]
    print(f"valid in both             : {len(both_valid)}")
    print(f"sum Δloglik               : {delta.sum():+10.3f}")
    print(f"  (positive ⇒ MLE-fitted params win in aggregate)")
    print(f"mean Δloglik per trial    : {delta.mean():+10.4f}")
    print(f"median Δloglik per trial  : {delta.median():+10.4f}")
    print(f"trials MLE-fitted better  : {int((delta > 0).sum())}")
    print(f"trials Chisq-fitted better: {int((delta < 0).sum())}")
    print(f"trials tied (Δ=0)         : {int((delta == 0).sum())}")
    print()
    print(f"=== Top {top_n} trials where MLE-fitted wins by the most ===")
    display(both_valid.nlargest(top_n, "delta_loglik"))
    print(f"\n=== Top {top_n} trials where Chisq-fitted wins by the most ===")
    display(both_valid.nsmallest(top_n, "delta_loglik"))
    return cmp


def plot_loglik_scatter(mle_chisq_df, mle_mle_df):
    """Scatter of per-trial mle_loglik: chisq-fitted (x) vs mle-fitted (y).
    Points above y=x: MLE-fitted parameters fit that trial better."""
    import matplotlib.pyplot as plt
    cmp = compare_loglik_per_trial(mle_chisq_df, mle_mle_df)
    both_valid = cmp[cmp["mle_chisq_valid"] & cmp["mle_mle_valid"]]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(both_valid["mle_chisq_loglik"],
               both_valid["mle_mle_loglik"], s=8, alpha=0.4)
    lo = float(min(both_valid["mle_chisq_loglik"].min(),
                   both_valid["mle_mle_loglik"].min()))
    hi = float(max(both_valid["mle_chisq_loglik"].max(),
                   both_valid["mle_mle_loglik"].max()))
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
    ax.set_xlabel("per-trial mle_loglik (chisq-fitted params)")
    ax.set_ylabel("per-trial mle_loglik (mle-fitted params)")
    ax.set_title("Per-trial loglik: chisq vs mle parameters\n"
                 "(above diagonal ⇒ mle-fitted wins this trial)")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_delta_histogram(mle_chisq_df, mle_mle_df, bins=60):
    """Histogram of Δloglik per trial. Positive → MLE-fitted wins."""
    import matplotlib.pyplot as plt
    cmp = compare_loglik_per_trial(mle_chisq_df, mle_mle_df)
    both_valid = cmp[cmp["mle_chisq_valid"] & cmp["mle_mle_valid"]]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(both_valid["delta_loglik"], bins=bins)
    ax.axvline(0.0, color="k", ls="--")
    ax.set_xlabel("Δ mle_loglik (mle-fitted − chisq-fitted)")
    ax.set_ylabel("count")
    ax.set_title("Per-trial loglik difference")
    fig.tight_layout()
    return fig

## Driver — specify subject + variant pair, produce both DFs

Set `SUBJECT_NAME`, then pick the chisq variant and the mle variant
(usually the same drift/bias/noise family in both, so they're directly
comparable). The cell assigns these globals for downstream cells:

- `mle_chisq_df`, `mle_chisq_eval`, `mle_chisq_config`, `mle_chisq_params`
- `mle_mle_df`,   `mle_mle_eval`,   `mle_mle_config`,   `mle_mle_params`

`MLE_LAPSE_RATE` controls the λ used in the re-evaluation:

- `None` (default): use each fit's baked-in value. Chisq fits don't fit
  `LAPSE_RATE`, so chisq → λ=0. MLE fits return the DE-found λ.
- `0.0`: disable lapse for both re-evaluations.
- e.g. `0.02`: apply the same fixed λ to both for an apples-to-apples
  comparison (gives chisq params the same outlier-protection the MLE
  fit had — clarifies whether MLE wins on signal or just on robustness).

In [ ]:
#SUBJECT_NAME = "FILL_ME_IN"
#SUBJECT_NAME = "Avgat3"
SUBJECT_NAME = "Rbp4_M2_1"
# Pair these so they share the same drift/bias/noise family for a
# like-for-like comparison.
#VARIANT_BASE = "Classic_biasNone__Normal(0, 1)_3s_dt0.005"
# VARIANT_BASE = "NoiseGain-RewardRate_biasQ-Val (Offset)_Normal(0, 1)_3s_dt0.005"
VARIANT_BASE = "Classic_biasQ-Val (Offset)_Normal(0, 1)_3s_dt0.005"
CHISQ_VARIANT_FILENAME = f"chisq_{VARIANT_BASE}.pkl"
MLE_VARIANT_FILENAME   = f"mle_{VARIANT_BASE}.pkl"
# 0.99 gives no-choice trials the survival-mass treatment
# (closer to what Chisq implicitly does). Set 0.0 to floor no-choice
# likelihoods to LOGLIK_FLOOR.
# MLE_TERMINAL_C = 0.99
MLE_TERMINAL_C = 0
# Lapse-rate override (λ) for both re-evaluations:
#   - None: use the value baked into each fit's params dict (chisq → 0,
#           mle → DE-found λ).
#   - 0.0:  disable lapse on both sides.
#   - 0.02: apply the same λ to both for an apples-to-apples comparison.
# MLE_LAPSE_RATE = None
MLE_LAPSE_RATE = None


if SUBJECT_NAME == "FILL_ME_IN":
    print("Set SUBJECT_NAME (and optionally the variant filenames) above, "
          "then re-run this cell.\n")
    print("Chisq variants:")
    for v in list_variants(fit_results_by_mode, "chisq"):
        print("  -", v)
    print("MLE variants:")
    for v in list_variants(fit_results_by_mode, "mle"):
        print("  -", v)
else:
    print(f"Subject     : {SUBJECT_NAME}")
    print(f"terminal_c  : {MLE_TERMINAL_C}")
    lapse_display = (
        f"{MLE_LAPSE_RATE}" if MLE_LAPSE_RATE is not None
        else "<from each fit>")
    print(f"lapse_rate  : {lapse_display}\n")
    (mle_chisq_df, mle_chisq_eval, mle_chisq_config,
     mle_chisq_params) = _evaluate_one(
        SUBJECT_NAME, CHISQ_VARIANT_FILENAME, "chisq-fitted",
        df_behavior, fit_results_by_mode, MLE_TERMINAL_C, MLE_LAPSE_RATE)
    (mle_mle_df, mle_mle_eval, mle_mle_config,
     mle_mle_params) = _evaluate_one(
        SUBJECT_NAME, MLE_VARIANT_FILENAME, "mle-fitted",
        df_behavior, fit_results_by_mode, MLE_TERMINAL_C, MLE_LAPSE_RATE)
    dn = mle_mle_eval.neg_loglik - mle_chisq_eval.neg_loglik
    print(f"Δ neg_loglik (mle-fitted - chisq-fitted): {dn:+.3f}")
    print("  (negative ⇒ MLE-fitted parameters fit better under MLE)")


## Inspection helpers

All helpers are functions that take a dataframe (or two) as input — so
the same code works on either `mle_chisq_df` or `mle_mle_df`, and the
comparison helpers operate on both at once.

## Inspect each DF individually

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ..model.mle_reeval import plot_rt_hist_colored_by_loss, plot_loss_distribution

def debug(mle_df):
    mle_df = mle_df[mle_df.valid == True]
    print(f"Sum of MLE log-likelihood: {mle_df.mle_loglik.sum():,.3f}")
    CUT_OFF = -40
    outlier_mask = mle_df.mle_loglik < CUT_OFF
    print(f"Number of outlier trials (loglik <= {CUT_OFF}): {(outlier_mask).sum():,}/{len(mle_df):,}")
    print(f"Sum of MLE log-likelihood without outliers: {mle_df.loc[~outlier_mask].mle_loglik.sum():,.3f}")
    # RT hist coloured by loss + per-trial -loglik distribution come from
    # model/mle_reeval.py (shared with rlmodel_compare.ipynb's grid rows).
    fig, ax = plt.subplots()
    plot_rt_hist_colored_by_loss(ax, mle_df, cut_off=CUT_OFF, t_dur=3.0)
    plt.show()
    fig, ax = plt.subplots()
    plot_loss_distribution(ax, mle_df, ylim=850)
    plt.show()
    # display(outliers_df.head(50))
    display(mle_df[["SessId", #"valid",
                    "TrialNumber", "DV", "ChoiceLeft", "ChoiceCorrect", "EarlyWithdrawal",
                    "PrevChoiceCorrect", "PrevOutcomeCount", "RewardRate5",
                     "calcStimulusTime", "mle_z", "mle_mu", "mle_sigma",
                    "mle_decision_time_observed","mle_choice_prob_or_density","mle_loglik","mle_valid_for_loss"]]
                    .sort_values("mle_loglik").head(20))
    #import matplotlib.pyplot as plt
    #_ = plt.hist(mle_df.mle_loglik, bins=60)
    total_len = len(mle_df)
    less_than_li = [-np.inf,
                    -700, -600, -500, -400, -300, -200, -100,
                    -90, -80, -70, -60, -50, -40, -30, -20, -10,
                    -9, -8, -7, -6, -5, -4, -3, -2, -1, 0,
                    1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
                    np.inf]
    last_num_digits = len(str(less_than_li[0]))
    last_limit = -np.inf
    prcnt_so_far = 0
    for threshold in less_than_li:
        num_digits = len(str(threshold))
        if last_num_digits != num_digits:
            print()  # Add a blank line when we cross an order of magnitude
            last_num_digits = num_digits
        count = (mle_df.mle_loglik <= threshold).sum()
        if last_limit == count:
            continue
        count_this_bin = count - (last_limit if np.isfinite(last_limit) else 0)
        last_limit = count
        prcnt = 100 * count_this_bin / total_len
        prcnt_so_far += prcnt
        print(f"Trials with loglik <= {threshold:>6}: {count_this_bin:,} / {total_len:,} "
              f"({prcnt:.2f}%)")
    print(f"Total percentage accounted for: {prcnt_so_far:.2f}%")

# print("Chisq-fitted parameters:")
# debug(mle_chisq_df)

print("\nMLE-fitted parameters:")
debug(mle_mle_df)

In [ ]:
def inspect_single(mle_df, label):
    print("=" * 60)
    print(f"=== {label} params, evaluated under MLE ===")
    print("=" * 60)
    print("\nTop-20 worst trials:")
    display(summarize_trial_losses(mle_df))
    print("\nMean loglik by RT bin (choice trials):")
    display(loss_by_rt_bin(mle_df))
    print("\nMean loglik by DV bin (choice trials):")
    display(loss_by_dv(mle_df))
    plot_trial_loss_histogram(mle_df, title_suffix=f" — {label}")


if "mle_chisq_df" not in globals() or "mle_mle_df" not in globals():
    print("No `mle_chisq_df` / `mle_mle_df` yet — run the driver cell above.")
else:
    inspect_single(mle_chisq_df, "CHISQ-FITTED")
    print("\n")
    inspect_single(mle_mle_df, "MLE-FITTED")

## Compare per-trial losses: chisq-fitted vs mle-fitted

Produces `cmp_df` (the merged comparison frame) as a global so you can
filter / sort / plot it further. Positive `delta_loglik` ⇒ MLE-fitted
parameters fit that trial better; negative ⇒ Chisq-fitted parameters
fit better.

In [ ]:
if "mle_chisq_df" not in globals() or "mle_mle_df" not in globals():
    print("No `mle_chisq_df` / `mle_mle_df` yet — run the driver cell above.")
else:
    cmp_df = summarize_comparison(mle_chisq_df, mle_mle_df)
    plot_delta_histogram(mle_chisq_df, mle_mle_df)
    plot_loglik_scatter(mle_chisq_df, mle_mle_df)